In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import csv
import plotly.graph_objects as go
import bisect
import random

import json
from collections import Counter
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from tqdm import tqdm
from time import time
from datetime import datetime

from input_run import data_payload_recomposition, CH_and_SH_decomposition, payload_alignment
from input_run import data_payload_recomposition_plus_Cert, recomp_plus_Cert, service_label
from input_run import Dataset_threshold_variations, Dataset_splitted, MATEC_data, BGRUA_data, EA_data
from input_run import labels_to_tensor, labels_to_int, data_payload_decomposition, balance_train
from input_run import payload_data_exctraction, train_validation_test_split, CH_payload_decomp, SH_payload_decomp 
from input_run import dataset_rec_ext_sni_pad_lengths, parameters_position, CH_view, payload_data_exctraction_with_PS_features, get_feature_importance_w_parameter_names
from classification_reports import classfication_report, misclassifications, misclassification_pairs, short_report, precision_recall_fscore_support

from Classifiers import BiGRUA_Classifier, MATEC_Classifier, EA_Classifier




In [ ]:
def sizes_equal(row):
    for i in range(len(row['Sizes'])):
        if(row['Sizes'][i]<700 and row['Is_uplink'][i] == 0):
            row['Sizes'][i] = 700
    return row

In [ ]:
def PacketSlicer(row, N):
    row['Sizes'] = row['Sizes'].strip('[]').split()[:N]
    row['Sizes'] = [int(num) for num in row['Sizes']]
    row['Is_uplink'] = row['Is_uplink'].strip('[]').split()[:N]
    row['Is_uplink'] = [int(num) for num in row['Is_uplink']]
    row['Times'] = row['Times'].strip('[]').split()[:N]
    row['Times'] = [float(num) for num in row['Times']]
    return row

In [ ]:
def flow_features(row, packets_num: int) -> np.array:
    sizes = np.array(row['Sizes'][:packets_num])
    times = np.array(row['Times'][:packets_num])
    is_uplink = np.array(row['Is_uplink'][:packets_num])
    uplink_index = [i for i in range(len(is_uplink)) if is_uplink[i] == 1]
    downlink_index = [i for i in range(len(is_uplink)) if is_uplink[i] == 0]

    uplink_sizes = sizes[uplink_index]
    downlink_sizes = sizes[downlink_index]

    uplink_times = times[uplink_index]
    uplink_intervals = (uplink_times[1:]-uplink_times[:-1])*1000
    

    downlink_times = times[downlink_index]
    downlink_intervals = (downlink_times[1:]-downlink_times[:-1])*1000

    if len(uplink_sizes) == 0:
        uplink_sizes = [0]
    if len(downlink_sizes) == 0:
        downlink_sizes = [0]

    if len(uplink_intervals) == 0:
        uplink_intervals = [0]
    if len(downlink_intervals) == 0:
        downlink_intervals = [0]

    return pd.DataFrame({
            'UL_num': len(uplink_sizes), 'DL_num': len(downlink_sizes),
            'UL_PS_max': max(uplink_sizes), 'UL_PS_min': min(uplink_sizes), 'UL_PS_ave': np.mean(uplink_sizes), 'UL_PS_std': np.std(uplink_sizes), 
            'UL_PS_25th': np.percentile(uplink_sizes, 25), 'UL_PS_50th': np.percentile(uplink_sizes, 50), 'UL_PS_75th': np.percentile(uplink_sizes, 75), 
            'DL_PS_max': max(downlink_sizes), 'DL_PS_min': min(downlink_sizes), 'DL_PS_ave': np.mean(downlink_sizes), 'DL_PS_std': np.std(downlink_sizes), 
            'DL_PS_25th': np.percentile(downlink_sizes, 25), 'DL_PS_50th': np.percentile(downlink_sizes, 50), 'DL_PS_75th': np.percentile(downlink_sizes, 75),
            'UL_TI_max': max(uplink_intervals), 'UL_TI_min': min(uplink_intervals), 'UL_TI_ave': np.mean(uplink_intervals), 'UL_TI_std': np.std(uplink_intervals), 
            'UL_TI_25th': np.percentile(uplink_intervals, 25), 'UL_TI_50th': np.percentile(uplink_intervals, 50), 'UL_TI_75th': np.percentile(uplink_intervals, 75), 
            'DL_TI_max': max(downlink_intervals), 'DL_TI_min': min(downlink_intervals), 'DL_TI_ave': np.mean(downlink_intervals), 'DL_TI_std': np.std(downlink_intervals), 
            'DL_TI_25th': np.percentile(downlink_intervals, 25), 'DL_TI_50th': np.percentile(downlink_intervals, 50), 'DL_TI_75th': np.percentile(downlink_intervals, 75)      
    }, index = [0])

In [ ]:
FLOW_STAT_PATH = ""
PAYLOAD_PATH = ""

In [ ]:
def parse_flow(s):
    if isinstance(s, str):
        try:
            return np.array([int(x) for x in s.split(' ') if x.strip() != ''], dtype=np.uint8)
        except Exception:
            return np.array([], dtype=np.uint8)
    return np.array([], dtype=np.uint8)

In [ ]:
TSPS = pd.read_csv(FLOW_STAT_PATH)
TSPS = TSPS.apply(lambda row: PacketSlicer(row, 15), axis=1)

In [ ]:
SNI = [TSPS.loc[i,'SNI'] for i in range(len(TSPS))]
sni_list = ['hireserve.com','www.oxilion.nl', 'artsandculture.google.com', 'www.zellis.com', 'planogram.scania.com', 'cec.konicaminolta.eu', 'cpireland.crowneplaza.com', 'coptrz.com', 'krystal.io', 'eduspot.co.uk']
TSPS =  TSPS[TSPS['SNI'].isin(sni_list)]
TSPS =  TSPS[TSPS['L4_protocol'] == 'QUIC']
TSPS = TSPS.reset_index(drop=True)
TSPS = TSPS.reset_index(drop=True)
features_list = TSPS.apply(lambda row: flow_features(row, len(row['Is_uplink'])), axis=1)
features_df = pd.concat(features_list.tolist(), ignore_index=True)
features_df['labels'] = TSPS['SNI']
features_df['pcap_name'] = TSPS['pcap_name']



In [ ]:
payload = pd.read_csv(PAYLOAD_PATH)
payload["Flow"] = payload["Flow"].apply(parse_flow)
features_df = features_df[features_df['pcap_name'].isin(payload['pcap_name'])]
features_df = features_df.reset_index(drop = True)
features_df['payload'] = payload['Flow']
features_df = features_df.drop('pcap_name', axis = 1)

In [ ]:
cols = features_df.columns.tolist()
cols[-1], cols[-2] = cols[-2], cols[-1]


features_df = features_df[cols]
features_df2 = features_df[[features_df.columns[-2]] + features_df.columns[:-2].tolist() + [features_df.columns[-1]]]
Data = features_df2[features_df2.columns[:-1]].to_numpy()


In [ ]:
labels = features_df2[features_df2.columns[-1]].to_numpy()
unique = Counter(labels)
print(unique)
unique_labels = list(set(labels))
labels_num = {value: idx for idx, value in enumerate(unique_labels)}
labels_int = [labels_num[label] for label in labels]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(Data, labels_int, 
                                                                              test_size=0.3)

X_train_p, X_train_f = [np.array(X_train[i][0]) for i in range(len(X_train))], [np.array(X_train[i][1:]) for i in range(len(X_train))]
X_test_p, X_test_f = [np.array(X_test[i][0]) for i in range(len(X_test))], [np.array(X_test[i][1:]) for i in range(len(X_test))]

X_train_pf = []
X_test_pf = []
for i in range(len(X_train)):
    try:
        X_train_pf.append(np.concatenate((X_train_p[i],X_train_f[i])))
    except:
        print('zero array')
for i in range(len(X_test)):
    try:
        X_test_pf.append(np.concatenate((X_test_p[i],X_test_f[i])))
    except:
        print('zero array')
rf  = RandomForestClassifier(n_jobs=10, n_estimators = 300, random_state=1, max_features = 0.5, 
                             max_depth = None)

rf.fit(X_train_pf, y_train)

y_hrbrf_predict = rf.predict(X_test_pf)
labels = list(range(len(labels_num)))
classfication_report(y_test, y_hrbrf_predict, labels_num)